# DSPy Basics

DSPy core concepts:

- **LM**

Can be called directly if we would like to:
```python
lm = dspy.LM('openai/gpt-4o')
print(lm("Where is Paris? Answer in one word"))
```

- **Signature**. DPSpy takes a signature and turns it into a prompt. String-based example of a signature is `'question, context -> reasoning, answer`. There is also a class-based definition of a signature (inherit from `dspy.Signature`)

- **Module** - the prompting technique we are going to use to ask LMs to do the task: `ChainOfThought`, `ProgramOfThought`, `MultipleChainComparison` etc. `Predict` is one the simplest one. Each instance of a module we create is referred to as a **program**.

```python
prog = dspy.Predict("poem, new_author -> rewritten_poem, explanation")
```

- **Prediction** - Python objects that represent the output of the LM in the format specified by the signature

```python
prediction = prog(poem="In silence I have watched you comb your hair.", new_author="Allen Ginsberg")
```

Signature = what we want to do; Module = how the LMs will be asked to do this, what _prompting techniques_ to use.

In [1]:
import os
from pprint import pprint

import dspy

## Basic Example

In [2]:
# Simple example
lm = dspy.LM(
    model="openrouter/google/gemma-4-31b-it:free",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
#
# Note: if you repeatedly call DSPy with the same input for the same LM, you’ll get the same output every time,
# all extremely quickly - it caches the providers’ responses.
# 
# dspy.configure_cache(
#     enable_disk_cache=False,
#     enable_memory_cache=False
# )
dspy.settings.configure(lm=lm)

In [3]:
# `lm` object can be used directly to make calls to an LLM. We can control cache per-call too.
# print(lm("Where is Paris? Answer in one word"))  # , cache=False)

In [4]:
# We can also provide a full conversation.
# print(lm(messages=[
#     {
#     "role": "system", 
#     "content": "You are a helpful assistant."
#     }, 
#     {               
#     "role": "user", 
#     "content": "Where is Ottawa? Answer in one word"
#     },
#     {
#     "role": "assistant",
#     "content": "Canada"
#     },
#     {
#     "role": "user", 
#     "content": "Where is Paris? Answer in one word"
#     }
#     ]))

In [5]:
predictor = dspy.Predict(
    "question: str, context: str -> answer: str, confidence: float"
)

In [6]:
# We can also use context to control multiple LLMs:
# with dspy.context(lm=lm_mini):
#     prediction = prog(question='Write a 5-line poem about toasters') 
#     print(prediction)
 
prediction = predictor(question="What is the capital of California?", context="")
pprint(prediction)

Prediction(
    answer='Sacramento',
    confidence=1.0
)


In [7]:
# "predictor" is a module
predictor = dspy.ChainOfThought(
    "question: str, context: str -> answer: str, confidence: float"
)
# "prediction" is "prediction" :)
prediction = predictor(question="What would be the capital of Ingria?", 
                       context="You can aspeculate about the future")
pprint(prediction)

Prediction(
    reasoning='Ingria (Ingermanland) is a historical region centered around the Gulf of Finland. In most historical or hypothetical scenarios involving an independent Ingrian state, the city of Saint Petersburg (historically founded in the heart of Ingria) would be the most logical choice for a capital due to its infrastructure, size, and strategic location. However, if the scenario assumes a separation from Russia where Saint Petersburg remains Russian, a smaller city like Gatchina or Luga might be considered. Given the prompt allows for speculation, I will identify Saint Petersburg as the primary candidate, as it is the central urban hub of the region.',
    answer='Based on geographical and historical context, the capital of a hypothetical or future state of Ingria would most likely be Saint Petersburg, as it is the largest city and the central hub of the Ingrian region.',
    confidence=0.8
)


## Basic Classification Example

The prompt we are going to re-implement:

```text
prompt = '''You are an expert customer intent classifier. Your goal is to classify customer messages into one of x predefined classes.
If you are not sure, return Unknown.
 
Labels:
1. Cancel Subscription
2. Update Email
3. Refund Request
4. Bug Report
5. Unknown
 
The output should be a JSON with the following structure:
{reasoning: "", intent: ""}
 
---------
Example:
Customer message: I want to end my subscription
 
---------
Example Output:
{Reasoning: "The main object is subscription and the main action is ending hence the user wants to cancel his subscription"
Intent: "Cancel Subscription"}
 
---------
Customer Message: My new email address is bob.smith@domain.com
Intent: 
'''
```

In [8]:
# "classifier" is a module with a simple signature.
# While ChainOfThought if powerful prompting technique, we can do even better and we can optimize it.
# We use "BootstrapFewShot" optimizer. Optimizers usually have their unique parameters. For example,
# BootstrapFewShot needs a a metric to hill climb and how max_labeled_demos.

classifier = dspy.ChainOfThought('message: str, labels: str -> intent_label: str')

labels = ['Cancel Subscription', 'Update Email', 'Refund Request',
          'Bug Report', 'Unknown']

examples = [dspy.Example({'message': 'I want to end my subscription',
                          'labels': labels,
                          'intent_label': 'Cancel Subscription'},
                         ).with_inputs('message', 'labels'),
            dspy.Example({'message': 'I deserve a refund',
                          'labels': labels,
                          'intent_label': 'Refund Request'}
                         ).with_inputs('message', 'labels'),
            dspy.Example({'message': 'The login screen is frozen, I cant login',
                          'labels': labels,
                          'intent_label': 'Bug Report'}
                         ).with_inputs('message', 'labels')
            ]

def msg_to_intent_metric(example, pred, trace=None):
    return pred.intent_label == example.intent_label

# The optimizer tunes the prompt for any model you pick, so a small, cheap model
# can often match or beat a hand-prompted frontier one.
# BootstrapFewShot composes a set of demos/examples to go into a predictor’s prompt.
# These demos come from a combination of labeled examples in the training set, and bootstrapped demos.
optimizer = dspy.BootstrapFewShot(metric=msg_to_intent_metric, max_labeled_demos=2)

In [9]:
# We can now take out classifier and optimize it!
classifier = optimizer.compile(classifier, trainset=examples)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 130.21it/s]

Bootstrapped 3 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.


In [10]:
prediction = classifier(
    message="My new email address is bob.smith@domain.com", labels=labels)
print(prediction)

Prediction(
    reasoning='The user is providing a new email address, which indicates an intent to change or update their contact information in the system.',
    intent_label='Update Email'
)


# Classification with 

In this example we use https://huggingface.co/datasets/tuetschek/atis - intent classification problem for airline inquiries.

In [11]:
from datasets import load_dataset

In [12]:
ds = load_dataset("tuetschek/atis")

In [13]:
ds.set_format(type='pandas')
df_train = ds['train'][:]
df_test = ds['test'][:]

In [14]:
df_test

,id,intent,text,slots
0,0,flight,i would like to find a flight from charlotte t...,O O O O O O O O B-fromloc.city_name O B-toloc....
1,1,airfare,on april first i need a ticket from tacoma to ...,O B-depart_date.month_name B-depart_date.day_n...
2,2,flight,on april first i need a flight going from phoe...,O B-depart_date.month_name B-depart_date.day_n...
3,3,flight,i would like a flight traveling one way from p...,O O O O O O B-round_trip I-round_trip O B-from...
4,4,flight,i would like a flight from orlando to salt lak...,O O O O O O B-fromloc.city_name O B-toloc.city...
...,...,...,...,...
888,888,flight,please find all the flights from cincinnati to...,O O O O O O B-fromloc.city_name O O O O O B-to...
889,889,flight,find me a flight from cincinnati to any airpor...,O O O O O B-fromloc.city_name O O O O O B-tolo...
890,890,flight,i 'd like to fly from miami to chicago on amer...,O O O O O O B-fromloc.city_name O B-toloc.city...
891,891,flight,i would like to book a round trip flight from ...,O O O O O O B-round_trip I-round_trip O O B-fr...


In [15]:
print(df_test['intent'].value_counts())

intent
flight               632
airfare               48
airline               38
ground_service        36
abbreviation          33
capacity              21
airport               18
flight+airfare        12
distance              10
aircraft               9
flight_no              8
ground_fare            7
city                   6
meal                   6
quantity               3
day_name               2
flight_time            1
airfare+flight         1
flight+airline         1
flight_no+airline      1
Name: count, dtype: int64


In [16]:
print(df_test.head(1)[['text', 'intent']])

                                                text  intent
0  i would like to find a flight from charlotte t...  flight


An LLM might find column names confusing. It does not matter for classical ML, but LMs have some prior knowledge about the world. Hence, let's rename the columns. To simplify the problem, we’ve also commented out any intents that are actually combinations of multiple intents:

In [17]:
ATIS_INTENT_MAPPING = {
   'abbreviation':                "Abbreviation and Fare Code Meaning Inquiry",
   'aircraft':                    "Aircraft Type Inquiry",
  #'aircraft+flight+flight_no':   "",
   'airfare':                     "Airfare and Fees Questions",
  #'airfare+flight_time':         "",
   'airline':                     "Airline Information Request",
  #'airline+flight_no':           "",
   'airport':                     "Airport Information and Queries",
   'capacity':                    "Aircraft Seating Capacity Inquiry",
   'Cheapest':                    "Cheapest Fare Inquiry",
   'city':                        "Airport Location Inquiry",
   'distance':                    "Airport Distance Inquiry",
   'flight':                      "Flight Booking Request",
  #'flight+airfare':              "",
   'flight_no':                   "Flight Number Inquiry",
   'flight_time':                 "Time Inquiry",
   'ground_fare':                 "Ground Transportation Cost Inquiry",
   'ground_service':              "Ground Transportation Inquiry",
  #'ground_service+ground_fare':  "Airport Ground Transportation and Cost Query",
   'meal':                        "Inquiry about In-flight Meals",
   'quantity':                    "Flight Quantity Inquiry",
   'restriction':                 "Flight Restriction Inquiry"
}
# Map the intents to their new names
df_test['intent'] = df_test['intent'].map(ATIS_INTENT_MAPPING)
df_test = df_test.dropna(subset=['intent'])
print(f"DF number of rows after removing multi label classes: {len(df_test)}")

DF number of rows after removing multi label classes: 876


In [18]:
unique_intents = df_test['intent'].unique()

Let's start from a simple yet useful implementation

In [19]:
# "one or more" makes the classifier to be multi-intent
class MultiIntentSignature(dspy.Signature):
   """
   Classify the message into one or more of the possible intent labels.
   """
   message: str      = dspy.InputField()
   labels: list[str] = dspy.InputField()
   intent_label: list[str] = dspy.OutputField(desc="Return the closest matches if any are "
      "reasonably close. Return as many as are close. Otherwise return None")

In [20]:
labels = list(unique_intents)
prog_classifier = dspy.Predict(MultiIntentSignature)

In [21]:
message = "I'd like to book a flight to Madrid from Berlin and order the pasta dinner."
prediction = prog_classifier(message=message, labels=labels)
print(prediction)

Prediction(
    intent_label=['Flight Booking Request', 'Inquiry about In-flight Meals']
)


In [22]:
# We can check what DSPy actually sent to a model:
lm.inspect_history(n=1)
# "Classify the message into one of the possible labels." becomes a part of the template, see "[[ ## completed ## ]]" section.





[2026-06-14T09:32:08.598104]

System message:

Your input fields are:
1. `message` (str): 
2. `labels` (list[str]):
Your output fields are:
1. `intent_label` (list[str]): Return the closest matches if any are reasonably close. Return as many as are close. Otherwise return None
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## message ## ]]
{message}

[[ ## labels ## ]]
{labels}

[[ ## intent_label ## ]]
{intent_label}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the message into one or more of the possible intent labels.


User message:

[[ ## message ## ]]
I'd like to book a flight to Madrid from Berlin and order the pasta dinner.

[[ ## labels ## ]]
["Flight Booking Request", "Airfare and Fees Questions", "Ground Transportation Inquiry", "Inquiry about In-flight Meals

In [24]:
prediction.get_lm_usage()  # Can be empty if served from cache.